# 과제 1: California Housing 데이터를 사용해 세 가지 Boosting 알고리즘  성능 비교

AdaBoostRegressor ,GradientBoostingRegressor,XGBRegressor

Regressor 연속형 숫자 예측 -> 평가 : MSE, RMSE, R² 

In [65]:
from sklearn.datasets import fetch_california_housing
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, confusion_matrix
import numpy as np

In [66]:
california = fetch_california_housing()
X = california.data 
y = california.target 

In [67]:
X.shape

(20640, 8)

In [68]:
y.shape

(20640,)

In [69]:
X_train, X_test, y_train, y_test = train_test_split (X,y,test_size=0.2, random_state=42)

## AdaBoostRegressor 

In [76]:
ab = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=1),
    n_estimators=200,
    learning_rate=0.05,
    random_state=42)

ab.fit(X_train, y_train)
ab_y_pred = ab.predict(X_test)

ab_r2 = r2_score(y_test, ab_y_pred)
ab_rmse = np.sqrt(mean_squared_error(y_test, ab_y_pred))

print("="*60)
print("AdaBoost 성능 평가")
print("="*60)
print(f"AdaBoostRegressor R²  : {ab_r2:.4f}")
print(f"AdaBoostRegressor RMSE: {ab_rmse:.4f}")

AdaBoost 성능 평가
AdaBoostRegressor R²  : 0.3441
AdaBoostRegressor RMSE: 0.9271


# GradientBoostingRegressor

In [71]:
gb = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42)

gb.fit(X_train, y_train)
gb_y_pred = gb.predict(X_test)

gb_r2 = r2_score(y_test, gb_y_pred)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_y_pred))

print("="*60)
print("Gradient 성능 평가")
print("="*60)
print(f"GradientBoostingRegressor R²  : {gb_r2:.4f}")
print(f"GradientBoostingRegressor RMSE: {gb_rmse:.4f}")
print("="*60)

Gradient 성능 평가
GradientBoostingRegressor R²  : 0.7778
GradientBoostingRegressor RMSE: 0.5396


# XGBRegressor 

In [72]:
xgb = XGBRegressor(
    n_estimators=200,          # 트리 개수
    learning_rate=0.05,       # learning_rate (eta)
    max_depth=4,              # 트리 깊이
    subsample=0.8,            # 샘플 일부만 사용 (과적합 방지)
    colsample_bytree=0.8,     # 특성 일부만 사용
    objective="reg:squarederror",  # 회귀용 목적함수
    random_state=42,
    n_jobs=-1)                # CPU 여러 코어 사용
 
xgb.fit(X_train, y_train)
xgb_y_pred = xgb.predict(X_test)

xgb_r2 = r2_score(y_test, xgb_y_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_y_pred))

print("="*60)
print("XGB 성능 평가")
print("="*60)
print(f"XGBRegressor R²  : {xgb_r2:.4f}")
print(f"XGBRegressor RMSE: {xgb_rmse:.4f}")
print("="*60)

XGB 성능 평가
XGBRegressor R²  : 0.8039
XGBRegressor RMSE: 0.5069


# 과제 2: 과적합 제어 실험
## Breast Cancer 데이터 사용해서  GradientBoostingClassifier로 과적합 제어 파라미터의 영향 분석
learning_rate: [0.01, 0.05, 0.1, 0.5]

max_depth: [1, 3, 5, 7, None]

subsample: [0.5, 0.7, 0.9, 1.0]

범주형 훈련/검증 평가 : Accuracy, Precision, Recall, F1, ROC-AUC 

In [44]:
cancer = load_breast_cancer()
c_X, c_y = cancer.data, cancer.target

In [45]:
c_X.shape

(569, 30)

In [46]:
c_y.shape

(569,)

In [47]:
c_X_train, c_X_test, c_y_train, c_y_test =  train_test_split (c_X,c_y,test_size=0.2, random_state=42)

In [ ]:
learning_rate_list = [0.01, 0.05, 0.1, 0.5] # 학습률
max_depth_list     = [1, 3, 5, 7, None] # 최대깊이
subsample_list     = [0.5, 0.7, 0.9, 1.0] # 훈련 샘플 중 몇 %를 사용할지

results = [] # 하이퍼리터 조합별로 딕셔너리 형태로 하나씩 담아둘 리스트.

for lr in learning_rate_list:
    for depth in max_depth_list:
        for subs in subsample_list:
            print(f"하이퍼파라미터 조합: learning_rate={lr}, max_depth={depth}, subsample={subs}")

            clf = GradientBoostingClassifier(
                n_estimators=200, # 200개 트리
                learning_rate=lr, # 학습률
                max_depth=depth, # 트리의 최대 깊이
                subsample=subs, # 샘플 비율
                random_state=42)

            clf.fit(c_X_train, c_y_train)

            y_train_pred = clf.predict(c_X_train) # 훈련 데이터에 대한 예측 결과
            y_test_pred  = clf.predict(c_X_test) # 테스트 데이터에 대한 예측 결과

            train_acc = accuracy_score(c_y_train, y_train_pred)
            test_acc  = accuracy_score(c_y_test, y_test_pred)
            overfit_gap = train_acc - test_acc
            cm = confusion_matrix(c_y_test, y_test_pred)

            print(f"test acc  : {test_acc:.4f}")
            print(f"overfit_gap: {overfit_gap:.4f}")
            print("Confusion Matrix:")
            print(cm)
            
            results.append({
                "learning_rate": lr,
                "max_depth": depth,
                "subsample": subs,
                "train_acc": train_acc,
                "test_acc": test_acc,
                "overfit_gap": overfit_gap
            })

하이퍼파라미터 조합: learning_rate=0.01, max_depth=1, subsample=0.5
test acc  : 0.9561
overfit_gap: -0.0023
Confusion Matrix:
[[39  4]
 [ 1 70]]
하이퍼파라미터 조합: learning_rate=0.01, max_depth=1, subsample=0.7
test acc  : 0.9474
overfit_gap: 0.0065
Confusion Matrix:
[[38  5]
 [ 1 70]]
하이퍼파라미터 조합: learning_rate=0.01, max_depth=1, subsample=0.9
test acc  : 0.9474
overfit_gap: 0.0065
Confusion Matrix:
[[38  5]
 [ 1 70]]
하이퍼파라미터 조합: learning_rate=0.01, max_depth=1, subsample=1.0
test acc  : 0.9386
overfit_gap: 0.0087
Confusion Matrix:
[[37  6]
 [ 1 70]]
하이퍼파라미터 조합: learning_rate=0.01, max_depth=3, subsample=0.5
test acc  : 0.9561
overfit_gap: 0.0373
Confusion Matrix:
[[40  3]
 [ 2 69]]
하이퍼파라미터 조합: learning_rate=0.01, max_depth=3, subsample=0.7
test acc  : 0.9561
overfit_gap: 0.0373
Confusion Matrix:
[[40  3]
 [ 2 69]]
하이퍼파라미터 조합: learning_rate=0.01, max_depth=3, subsample=0.9
test acc  : 0.9561
overfit_gap: 0.0373
Confusion Matrix:
[[40  3]
 [ 2 69]]
하이퍼파라미터 조합: learning_rate=0.01, max_depth=3, subsample